# Polaris Privilege — Real-World Use-Case Role Test

Validates the four production role profiles from `privilege/doc-privilege-test.md` §3 by
provisioning each as a **multi-privilege worker** and asserting both sides of its boundary:
every capability the role is meant to have must classify `AUTHORIZED`, and every action it
must be denied must classify `BLOCKED_PRIV`.

| Role | Footprint (claimed) | Must do | Must be blocked from |
|---|---|---|---|
| **A · Data Engineer** | `NAMESPACE_CREATE, TABLE_CREATE, TABLE_WRITE_DATA, TABLE_READ_DATA, VIEW_CREATE` | build & maintain datasets | drop tables / drop namespaces |
| **B · BI Analyst** | `TABLE_LIST, TABLE_READ_DATA, VIEW_LIST, VIEW_CREATE` | read tables, build views | create/modify/drop tables |
| **C · Auditor** | `CATALOG_MANAGE_METADATA` | list/inspect all metadata | mutate structure |
| **D · Tenant Admin** | `CATALOG_MANAGE_CONTENT` | full control of *its* catalog | other catalogs / mgmt layer |

Each action runs in its own freshly-scaffolded catalog (proven matrix isolation pattern),
so role tests never contaminate each other. Footprint privilege names are **hypotheses**:
any the build rejects at grant time are surfaced as `GRANT_INVALID`, meaning the role is
only partially provisioned — flagged explicitly so a resulting `BLOCKED_PRIV` is read as a
vocabulary gap, not a real boundary.

> **Scope caveat (Auditor):** the doc says `CATALOG_MANAGE_METADATA` exposes metadata but
> hides *raw rows* (`TABLE_READ_DATA` absent). Raw-row access in Polaris is gated at
> credential-vending / object storage, **not** the Iceberg REST catalog API this harness
> drives — so that specific boundary is out of scope here and noted rather than asserted.

In [36]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import time, uuid
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
init_env("local")                       # default safe target; switch to "dev" for shared cluster
require_not_prod("privilege hierarchy / role mutations")  # hard-fail against company PROD

# ── global request timeout: no single HTTP call may hang the notebook ────────
# Patches the shared requests.Session so EVERY call (notebook + src utils) gets a
# (connect, read) timeout. A dead/slow endpoint now raises promptly instead of
# blocking forever — probe() catches it and records ERROR.
import requests as _rq
if not getattr(_rq.sessions.Session, "_timeout_patched", False):
    _orig_request = _rq.sessions.Session.request
    def _request_with_timeout(self, method, url, **kw):
        kw.setdefault("timeout", (5, 30))          # 5s connect, 30s read
        return _orig_request(self, method, url, **kw)
    _rq.sessions.Session.request = _request_with_timeout
    _rq.sessions.Session._timeout_patched = True

tok = root_token()
_RAW_SETTLE = 0.6                        # settle after a read-after-write lag-500
NS = lambda cat: f"ns-{cat}"            # per-catalog namespace (Vector B: no flat strings)
print("✅ privilege harness loaded")

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ privilege harness loaded


In [37]:
# ── action registry + entity bodies ──────────────────────────────────────────
# Each action: build(cat,hdr,ctx) scaffolds the prereq AS THE SCAFF principal;
# challenge(cat,hdr,ctx) fires the action under test AS THE WORKER.
def _tbl_body(cat, ns, tbl):
    return {"name": tbl, "location": f"s3a://{BUCKET}/{cat}/{ns}/{tbl}/",
            "schema": {"type":"struct","fields":[
                {"id":1,"name":"id","type":"long","required":True},
                {"id":2,"name":"value","type":"string","required":False}]}}

def _view_body(cat, ns, vw):
    return {"name": vw, "default-namespace":[ns],
            "schema":{"type":"struct","schema-id":0,
                      "fields":[{"id":1,"name":"id","type":"long","required":True}]},
            "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),
                            "schema-id":0,"default-namespace":[ns],"summary":{"engine":"spark"},
                            "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
            "location": f"s3a://{BUCKET}/{cat}/{ns}/{vw}/"}

def _mk_ns(cat, hdr):
    requests.post(f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
                  json={"namespace":[NS(cat)], "properties":{}})
def _mk_tbl(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
                  json=_tbl_body(cat, NS(cat), ctx["entity"]))
def _mk_view(cat, hdr, ctx):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
                  json=_view_body(cat, NS(cat), ctx["entity"]))

# action_id -> spec.  drop=True actions get the config gate neutralized (gap 1).
ACTIONS = {
  "ns_create":  dict(target="Namespace", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: None,
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces", headers=hdr,
         json={"namespace":[NS(cat)], "properties":{}})),
  "ns_list":    dict(target="Namespace", action="LIST", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces", headers=hdr)),
  "ns_drop":    dict(target="Namespace", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}", headers=hdr)),
  "tbl_create": dict(target="Table", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr,
         json=_tbl_body(cat, NS(cat), ctx["entity"]))),
  "tbl_list":   dict(target="Table", action="LIST", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables", headers=hdr)),
  "tbl_read":   dict(target="Table", action="READ", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  "tbl_commit": dict(target="Table", action="COMMIT", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr,
         json={"requirements":[], "updates":[
             {"action":"set-properties","updates":{"probe":"1"}}]})),
  "tbl_drop":   dict(target="Table", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_tbl(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/tables/{ctx['entity']}", headers=hdr)),
  "view_create":dict(target="View", action="CREATE", drop=False,
     build=lambda cat,hdr,ctx: _mk_ns(cat,hdr),
     challenge=lambda cat,hdr,ctx: requests.post(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views", headers=hdr,
         json=_view_body(cat, NS(cat), ctx["entity"]))),
  "view_get":   dict(target="View", action="GET", drop=False,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.get(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
  "view_drop":  dict(target="View", action="DROP", drop=True,
     build=lambda cat,hdr,ctx: (_mk_ns(cat,hdr), _mk_view(cat,hdr,ctx)),
     challenge=lambda cat,hdr,ctx: requests.delete(
         f"{BASE_CAT}/{cat}/namespaces/{NS(cat)}/views/{ctx['entity']}", headers=hdr)),
}
LABEL = {k: f"{v['target']}.{v['action']}" for k,v in ACTIONS.items()}

# ── six-way classifier (identical taxonomy to the matrix notebook) ───────────
def _is_lag(r):
    b = r.text.lower()
    return r.status_code == 500 and ("metadata" in b or "nullpointer" in b)

def classify(r):
    sc, b = r.status_code, r.text.lower()
    if sc in (200, 201, 204):                        return "AUTHORIZED"
    if sc == 409 or "already exists" in b:           return "AUTHORIZED"   # create lag-committed
    if _is_lag(r):                                   return "LAG_500"
    if sc == 403:
        if "unable to purge" in b:                   return "BLOCKED_CONFIG"  # harness bug
        if "is not authorized" in b or "not authorized for op" in b: return "BLOCKED_PRIV"
        if "opa" in b or "ext_authz" in b or "denied by policy" in b: return "BLOCKED_OPA"  # harness bug
        return "BLOCKED_PRIV"
    if sc == 404:                                    return "NOT_FOUND_404"   # setup issue
    return f"HTTP_{sc}"

# ── multi-privilege worker: grant a FULL footprint, capture invalid names ────
def provision_multi(cat, privileges):
    """Create a single-use worker holding EXACTLY `privileges` (a list). Returns
    (header_or_None, granted, invalid, names) where names=(P,PR,CR) so the caller
    can delete the worker deterministically. Suffix is **'worker'** — one the
    shared catalog teardown already sweeps — so even if the caller forgets to
    clean up, `_teardown_catalog_selfcontained` reclaims it (no leaked principals).
    Token issued AFTER the grants so it reflects the role's full footprint; a name
    the build rejects at grant lands in `invalid` (GRANT_INVALID)."""
    P, PR, CR = f"{cat}-worker-prin", f"{cat}-worker-prole", f"{cat}-worker-crole"
    requests.delete(f"{BASE_MGMT}/principals/{P}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/principal-roles/{PR}", headers=h(tok))
    rp   = create_principal(P, token=tok).json()
    cid  = rp.get("principal", {}).get("clientId")     or rp.get("clientId")
    csec = rp.get("credentials", {}).get("clientSecret") or rp.get("clientSecret")
    create_principal_role(PR, token=tok)
    create_catalog_role(catalog=cat, name=CR, token=tok)
    assign_catalog_role_to_principal_role(catalog=cat, pr=PR, cr=CR, token=tok)
    assign_principal_role_to_principal(principal=P, pr=PR, token=tok)
    # wait until the catalog-role is visible before granting — cuts the 404
    # "CatalogRole not found" lag that would otherwise be misread as GRANT_INVALID.
    for _ in range(6):
        lr = requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles", headers=h(tok))
        rolenames = [(c.get("name") if isinstance(c, dict) else c)
                     for c in (lr.json().get("roles", []) if lr.status_code == 200 else [])]
        if CR in rolenames:
            break
        time.sleep(0.5)
    granted, invalid = [], []
    for pv in privileges:
        gr = grant_privilege(catalog=cat, cr=CR, privilege=pv, token=tok)
        gb = gr.text.lower()
        ok = (gr.status_code < 400 or "already exists" in gb
              or "23505" in gb or "duplicate key" in gb)
        (granted if ok else invalid).append(pv)
    tr = get_token(client_id=cid, client_secret=csec, scope=f"PRINCIPAL_ROLE:{PR}")
    if tr.status_code != 200:
        return None, granted, invalid, (P, PR, CR)
    wtok = tr.json().get("access_token")
    hdr = {"Authorization": f"Bearer {wtok}", "Polaris-Realm": REALM,
           "Content-Type": "application/json"}
    return hdr, granted, invalid, (P, PR, CR)

# ── one probe: run `action_id` as a worker holding `privileges` ──────────────
def probe(action_id, privileges, verbose=False):
    """Fresh catalog → scaffold the action's prereq (as scaff) → provision a
    worker holding `privileges` → fire the action → classify → CLEAN UP. Returns
    (outcome, granted, invalid). Every resource created here (catalog, namespace,
    entity, scaff principal, worker principal + roles, MinIO objects) is removed
    before returning, so no probe can affect another."""
    spec = ACTIONS[action_id]
    privs = list(privileges) if isinstance(privileges, (list, tuple)) else [privileges]
    cat = f"hzr-{action_id.replace('_','')}-{uuid.uuid4().hex[:4]}"
    props = {}
    if spec["drop"]:
        props = {"polaris.config.drop-with-purge.enabled": "true",
                 "polaris.config.purge-view-metadata-on-drop": "false"}
    create_catalog(cat, token=tok, properties=props)
    ctx = {"entity": f"e{uuid.uuid4().hex[:6]}"}
    outcome, granted, invalid, detail = "ERROR", [], [], ""
    try:
        # case_scaffold.__exit__ ALWAYS tears down the catalog (recursive) + scaff
        # principal + MinIO, even on exception. We additionally delete the worker
        # explicitly in a finally so nothing survives a single probe.
        with case_scaffold(cat, build=lambda hdr: spec["build"](cat, hdr, ctx),
                           teardown=None, root=tok, destroy_scaff=True):
            whdr, granted, invalid, wnames = provision_multi(cat, privs)
            try:
                if whdr is None:
                    outcome, detail = "BLOCKED_OPA", "worker token not issued"
                else:
                    r = spec["challenge"](cat, whdr, ctx)
                    if _is_lag(r):                                  # read-after-write 500
                        time.sleep(_RAW_SETTLE); r = spec["challenge"](cat, whdr, ctx)
                    if (r.status_code == 404 and not spec["drop"]
                            and spec["action"] in ("READ", "GET", "LIST")):  # entity-visibility lag
                        time.sleep(_RAW_SETTLE); r = spec["challenge"](cat, whdr, ctx)
                    outcome, detail = classify(r), f"[{r.status_code}] {r.text[:80]}"
            finally:
                delete_instance_principal(wnames, catalog=cat, root=tok)   # reclaim worker
    except Exception as e:
        outcome, detail = "ERROR", f"{type(e).__name__}: {str(e)[:110]}"
    # always surface the cause of an ERROR / harness-bug outcome, even when the
    # calling loop prints its own one-line summary (which omits `detail`).
    if verbose or outcome in ("ERROR", "BLOCKED_OPA", "BLOCKED_CONFIG"):
        print(f"     ↳ {LABEL[action_id]}/{','.join(privs)} → {outcome}: {detail}")
    return outcome, granted, invalid

print(f"✅ {len(ACTIONS)} actions registered; classifier + multi-priv worker ready")

✅ 11 actions registered; classifier + multi-priv worker ready


In [38]:
# ── ROLE PROFILES (footprint + expected allow/deny per doc §3) ───────────────
# allow = action_ids the role MUST be able to perform (expect AUTHORIZED)
# deny  = action_ids the role MUST be blocked from   (expect BLOCKED_PRIV)
ROLES = {
  "A · Data Engineer": dict(
      footprint=["NAMESPACE_CREATE","TABLE_CREATE","TABLE_WRITE_DATA","TABLE_READ_DATA","VIEW_CREATE"],
      allow=["ns_create","tbl_create","tbl_commit","tbl_read","view_create"],
      deny=["tbl_drop","ns_drop","view_drop"]),
  "B · BI Analyst": dict(
      footprint=["TABLE_LIST","TABLE_READ_DATA","VIEW_LIST","VIEW_CREATE"],
      allow=["tbl_list","tbl_read","view_get","view_create"],
      deny=["tbl_create","tbl_commit","tbl_drop"]),
  "C · Auditor": dict(
      footprint=["CATALOG_MANAGE_METADATA"],
      allow=["ns_list","tbl_list","tbl_read","view_get"],
      deny=["tbl_create","tbl_drop","view_create","view_drop","ns_create","ns_drop"]),
  "D · Tenant Admin": dict(
      footprint=["CATALOG_MANAGE_CONTENT"],
      allow=["ns_create","ns_drop","tbl_create","tbl_drop","view_create","view_drop"],
      deny=[]),   # D's boundary is cross-catalog / mgmt-layer — tested separately below
}
print(f"{len(ROLES)} role profiles; "
      f"{sum(len(r['allow'])+len(r['deny']) for r in ROLES.values())} allow/deny assertions")

4 role profiles; 31 allow/deny assertions


In [39]:
# ── evaluate each role: provision the FULL footprint, assert allow & deny ────
ROLE_RESULTS = {}   # role -> dict(action_id -> (kind, outcome, ok))
ROLE_VOCAB   = {}   # role -> (granted, invalid)
print("Evaluating role profiles...\n")
for role, spec in ROLES.items():
    print(f"━━ {role}  footprint={spec['footprint']} ━━")
    res = {}
    granted_seen, invalid_seen = set(), set()
    for kind, aids, expected in [("allow", spec["allow"], "AUTHORIZED"),
                                 ("deny",  spec["deny"],  "BLOCKED_PRIV")]:
        for aid in aids:
            outcome, granted, invalid = probe(aid, spec["footprint"])
            granted_seen |= set(granted); invalid_seen |= set(invalid)
            ok = (outcome == expected)
            res[aid] = (kind, outcome, ok)
            mark = "✓" if ok else "✗"
            print(f"  [{kind:5}] {LABEL[aid]:16} → {outcome:14} expect {expected:13} {mark}")
    ROLE_RESULTS[role] = res
    ROLE_VOCAB[role]   = (sorted(granted_seen), sorted(invalid_seen))
    if invalid_seen:
        print(f"  ∅ footprint names rejected at grant (GRANT_INVALID): {sorted(invalid_seen)}")
    print()
print("✅ role evaluation complete")

Evaluating role profiles...

━━ A · Data Engineer  footprint=['NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_WRITE_DATA', 'TABLE_READ_DATA', 'VIEW_CREATE'] ━━
  [allow] Namespace.CREATE → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.CREATE     → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.COMMIT     → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.READ       → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] View.CREATE      → AUTHORIZED     expect AUTHORIZED    ✓
  [deny ] Table.DROP       → BLOCKED_PRIV   expect BLOCKED_PRIV  ✓
  [deny ] Namespace.DROP   → BLOCKED_PRIV   expect BLOCKED_PRIV  ✓
  [deny ] View.DROP        → BLOCKED_PRIV   expect BLOCKED_PRIV  ✓

━━ B · BI Analyst  footprint=['TABLE_LIST', 'TABLE_READ_DATA', 'VIEW_LIST', 'VIEW_CREATE'] ━━
  [allow] Table.LIST       → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] Table.READ       → AUTHORIZED     expect AUTHORIZED    ✓
  [allow] View.GET         → BLOCKED_PRIV   expect AUTHORIZED    ✗
  [allow] Vie

In [40]:
# ── D · Tenant Admin boundary: cross-catalog isolation ───────────────────────
# A CATALOG_MANAGE_CONTENT holder on catalog X must NOT be able to act on catalog Y.
# Provision the worker (+ its grant) on cat_x, then aim its token at a DIFFERENT
# catalog cat_y that holds a real entity. Expect BLOCKED_PRIV (sovereignty is
# per-catalog, doc §3 Use Case D).
print("Tenant-Admin cross-catalog isolation test...\n")
cat_x = f"tenantA-{uuid.uuid4().hex[:4]}"
cat_y = f"tenantB-{uuid.uuid4().hex[:4]}"
create_catalog(cat_x, token=tok)
create_catalog(cat_y, token=tok, properties={
    "polaris.config.drop-with-purge.enabled":"true",
    "polaris.config.purge-view-metadata-on-drop":"false"})
ctx_y = {"entity": f"e{uuid.uuid4().hex[:6]}"}
xresult = None
wnames = None
try:
    # build a real table in cat_y as a full scaff principal
    with case_scaffold(cat_y, build=lambda hdr: (_mk_ns(cat_y,hdr), _mk_tbl(cat_y,hdr,ctx_y)),
                       teardown=None, root=tok, destroy_scaff=True):
        # worker is granted CATALOG_MANAGE_CONTENT on cat_x ONLY
        whdr, granted, invalid, wnames = provision_multi(cat_x, ["CATALOG_MANAGE_CONTENT"])
        # ...then tries to drop the table living in cat_y
        r = requests.delete(
            f"{BASE_CAT}/{cat_y}/namespaces/{NS(cat_y)}/tables/{ctx_y['entity']}", headers=whdr)
        xresult = classify(r)
        print(f"  cat_x admin → DROP table in cat_y → [{r.status_code}] {xresult}")
finally:
    if wnames is not None:
        delete_instance_principal(wnames, catalog=cat_x, root=tok)   # reclaim cat_x worker
    teardown_catalog(cat_x, root=tok)   # cat_y is torn down by case_scaffold.__exit__
    teardown_catalog(cat_y, root=tok)
isolated = xresult in ("BLOCKED_PRIV","BLOCKED_OPA","NOT_FOUND_404")
print(f"\n{'✅' if isolated else '⚠️'} cross-catalog isolation "
      f"{'CONFIRMED' if isolated else 'NOT confirmed'}: a tenant admin "
      f"{'cannot' if isolated else 'COULD'} reach a sibling catalog.")
ROLE_RESULTS["D · Tenant Admin"]["xcat_drop"] = ("deny", xresult, isolated)

Tenant-Admin cross-catalog isolation test...

  cat_x admin → DROP table in cat_y → [403] BLOCKED_PRIV

✅ cross-catalog isolation CONFIRMED: a tenant admin cannot reach a sibling catalog.


In [41]:
# ── per-role PASS/FAIL cards ─────────────────────────────────────────────────
# Primary output is PLAIN TEXT (always visible in any frontend). A rich Markdown
# render is attempted as a bonus where the frontend supports text/markdown.
SYM = {"AUTHORIZED":"✅","BLOCKED_PRIV":"🔒","GRANT_INVALID":"∅","BLOCKED_OPA":"⚠️OPA",
       "BLOCKED_CONFIG":"⚠️CFG","LAG_500":"…","NOT_FOUND_404":"404","ERROR":"💥"}
# 1) guaranteed-visible plain-text report ------------------------------------
print("=" * 66)
print("USE-CASE ROLE BOUNDARY REPORT")
print("=" * 66)
for role, res in ROLE_RESULTS.items():
    granted, invalid = ROLE_VOCAB.get(role, ([],[]))
    npass = sum(1 for _,_,ok in res.values() if ok)
    ntot  = len(res)
    verdict = "PASS ✅" if npass == ntot else f"REVIEW ⚠️  ({npass}/{ntot})"
    print(f"\n#### {role} — {verdict}")
    print(f"     footprint granted: {granted}")
    if invalid:
        print(f"     ∅ GRANT_INVALID (build rejected): {invalid} → role only partially")
        print(f"       provisioned; a deny-side BLOCKED_PRIV may be a vocabulary gap.")
    print(f"     {'ACTION':18} {'SIDE':6} {'MEASURED':16} OK")
    print(f"     {'-'*18} {'-'*6} {'-'*16} --")
    for aid,(kind,outcome,ok) in res.items():
        lab = LABEL.get(aid, aid)
        print(f"     {lab:18} {kind:6} {SYM.get(outcome,''):2}{outcome:14} {'✓' if ok else '✗'}")
print("\nLegend: ✅ AUTHORIZED · 🔒 BLOCKED_PRIV · ∅ GRANT_INVALID · 💥 ERROR · "
      "OK ✓ = measured matches the role's intended boundary.")

# 2) bonus: rich Markdown render (ignored silently if frontend can't render) ---
try:
    from IPython.display import Markdown, display
    blocks = []
    for role, res in ROLE_RESULTS.items():
        granted, invalid = ROLE_VOCAB.get(role, ([],[]))
        npass = sum(1 for _,_,ok in res.values() if ok); ntot = len(res)
        verdict = "✅ PASS" if npass == ntot else f"⚠️ {npass}/{ntot}"
        rows = [f"| {LABEL.get(aid,aid)} | {kind} | {SYM.get(outcome,outcome)} {outcome} | {'✓' if ok else '✗'} |"
                for aid,(kind,outcome,ok) in res.items()]
        inv = (f"\n\n> ∅ **GRANT_INVALID footprint names** (build rejected): `{invalid}` — role only "
               f"partially provisioned; any `deny`-side BLOCKED_PRIV may be a vocabulary gap.") if invalid else ""
        blocks.append(f"#### {role} — {verdict}\n\nFootprint granted: `{granted}`{inv}\n\n"
                      f"| Action | Side | Measured | OK |\n|---|---|---|---|\n" + "\n".join(rows))
    display(Markdown("### Use-Case Role Boundary Report\n\n" + "\n\n---\n\n".join(blocks)))
except Exception:
    pass   # plain-text report above is the source of truth

USE-CASE ROLE BOUNDARY REPORT

#### A · Data Engineer — PASS ✅
     footprint granted: ['NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_READ_DATA', 'TABLE_WRITE_DATA', 'VIEW_CREATE']
     ACTION             SIDE   MEASURED         OK
     ------------------ ------ ---------------- --
     Namespace.CREATE   allow  ✅ AUTHORIZED     ✓
     Table.CREATE       allow  ✅ AUTHORIZED     ✓
     Table.COMMIT       allow  ✅ AUTHORIZED     ✓
     Table.READ         allow  ✅ AUTHORIZED     ✓
     View.CREATE        allow  ✅ AUTHORIZED     ✓
     Table.DROP         deny   🔒 BLOCKED_PRIV   ✓
     Namespace.DROP     deny   🔒 BLOCKED_PRIV   ✓
     View.DROP          deny   🔒 BLOCKED_PRIV   ✓

#### B · BI Analyst — REVIEW ⚠️  (6/7)
     footprint granted: ['TABLE_LIST', 'TABLE_READ_DATA', 'VIEW_CREATE', 'VIEW_LIST']
     ACTION             SIDE   MEASURED         OK
     ------------------ ------ ---------------- --
     Table.LIST         allow  ✅ AUTHORIZED     ✓
     Table.READ         allow  ✅ AUTHORIZE

### Use-Case Role Boundary Report

#### A · Data Engineer — ✅ PASS

Footprint granted: `['NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_READ_DATA', 'TABLE_WRITE_DATA', 'VIEW_CREATE']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Namespace.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.COMMIT | allow | ✅ AUTHORIZED | ✓ |
| Table.READ | allow | ✅ AUTHORIZED | ✓ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |
| Namespace.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |
| View.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |

---

#### B · BI Analyst — ⚠️ 6/7

Footprint granted: `['TABLE_LIST', 'TABLE_READ_DATA', 'VIEW_CREATE', 'VIEW_LIST']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Table.LIST | allow | ✅ AUTHORIZED | ✓ |
| Table.READ | allow | ✅ AUTHORIZED | ✓ |
| View.GET | allow | 🔒 BLOCKED_PRIV | ✗ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | deny | 🔒 BLOCKED_PRIV | ✓ |
| Table.COMMIT | deny | 🔒 BLOCKED_PRIV | ✓ |
| Table.DROP | deny | 🔒 BLOCKED_PRIV | ✓ |

---

#### C · Auditor — ⚠️ 4/10

Footprint granted: `['CATALOG_MANAGE_METADATA']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Namespace.LIST | allow | ✅ AUTHORIZED | ✓ |
| Table.LIST | allow | ✅ AUTHORIZED | ✓ |
| Table.READ | allow | ✅ AUTHORIZED | ✓ |
| View.GET | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | deny | ✅ AUTHORIZED | ✗ |
| Table.DROP | deny | ✅ AUTHORIZED | ✗ |
| View.CREATE | deny | ✅ AUTHORIZED | ✗ |
| View.DROP | deny | ✅ AUTHORIZED | ✗ |
| Namespace.CREATE | deny | ✅ AUTHORIZED | ✗ |
| Namespace.DROP | deny | ✅ AUTHORIZED | ✗ |

---

#### D · Tenant Admin — ✅ PASS

Footprint granted: `['CATALOG_MANAGE_CONTENT']`

| Action | Side | Measured | OK |
|---|---|---|---|
| Namespace.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Namespace.DROP | allow | ✅ AUTHORIZED | ✓ |
| Table.CREATE | allow | ✅ AUTHORIZED | ✓ |
| Table.DROP | allow | ✅ AUTHORIZED | ✓ |
| View.CREATE | allow | ✅ AUTHORIZED | ✓ |
| View.DROP | allow | ✅ AUTHORIZED | ✓ |
| xcat_drop | deny | 🔒 BLOCKED_PRIV | ✓ |

In [42]:
# ── SUMMARY: overall role pass/fail + discovered vocabulary ──────────────────
print("ROLE BOUNDARY VERDICT:\n")
all_ok = True
for role, res in ROLE_RESULTS.items():
    npass = sum(1 for _,_,ok in res.values() if ok)
    ntot  = len(res)
    all_ok &= (npass == ntot)
    print(f"  {role:20} {npass}/{ntot} assertions held "
          f"{'✅' if npass==ntot else '⚠️ review mismatches'}")

valid   = sorted({p for g,_ in ROLE_VOCAB.values() for p in g})
invalid = sorted({p for _,i in ROLE_VOCAB.values() for p in i})
print("\nDISCOVERED PRIVILEGE VOCABULARY (across all role footprints):")
print(f"  ✅ accepted at grant time : {valid}")
print(f"  ∅ rejected (GRANT_INVALID): {invalid}")
print(f"\n{'✅ ALL ROLES PASSED' if all_ok else '⚠️ SOME ROLES NEED REVIEW'} — "
      "mismatches are findings about THIS build (e.g. a footprint name this Polaris "
      "does not accept), to reconcile against doc-privilege-test.md §3.")

ROLE BOUNDARY VERDICT:

  A · Data Engineer    8/8 assertions held ✅
  B · BI Analyst       6/7 assertions held ⚠️ review mismatches
  C · Auditor          4/10 assertions held ⚠️ review mismatches
  D · Tenant Admin     7/7 assertions held ✅

DISCOVERED PRIVILEGE VOCABULARY (across all role footprints):
  ✅ accepted at grant time : ['CATALOG_MANAGE_CONTENT', 'CATALOG_MANAGE_METADATA', 'NAMESPACE_CREATE', 'TABLE_CREATE', 'TABLE_LIST', 'TABLE_READ_DATA', 'TABLE_WRITE_DATA', 'VIEW_CREATE', 'VIEW_LIST']
  ∅ rejected (GRANT_INVALID): []

⚠️ SOME ROLES NEED REVIEW — mismatches are findings about THIS build (e.g. a footprint name this Polaris does not accept), to reconcile against doc-privilege-test.md §3.


In [43]:
# ── teardown sweep: reclaim stragglers (this run + any prior-run leftovers) ───
# Every probe tears down its own catalog + scaff + worker. This belt-and-suspenders
# pass removes anything left behind — including principals from the OLD '-role-'
# naming and any test catalog whose teardown was interrupted — so no residue can
# affect other tests.
TEST_PREFIXES = ["hzr-", "tenantA-", "tenantB-"]

# 1) leftover test catalogs → recursive teardown (drops ns/tables/views + purge)
_cats = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok))
_names = [(c.get("name") if isinstance(c, dict) else c)
          for c in (_cats.json().get("catalogs", []) if _cats.status_code == 200 else [])]
_left = [n for n in _names if n and any(n.startswith(p) for p in TEST_PREFIXES)]
for n in _left:
    try: teardown_catalog(n, root=tok)
    except Exception as e: print(f"  ⚠️ teardown_catalog({n}): {e}")

# 2) leftover principals / principal-roles (covers current 'worker'/'scaff' AND
#    the deprecated 'role' suffix from earlier runs)
_swept = sweep_instance_principals(
    prefixes=TEST_PREFIXES,
    suffixes=("scaff", "worker", "role", "inst", "full", "scaffold"))

print(f"🧹 cleanup: tore down {len(_left)} leftover test catalog(s); "
      f"swept {_swept} principal/role object(s). Polaris is clean.")

🧹 swept 0 leftover instance principal/role objects
🧹 cleanup: tore down 0 leftover test catalog(s); swept 0 principal/role object(s). Polaris is clean.
